# Phase 2 — Frozen Evaluation Summary

**Read-only analysis.** Uses saved evaluation outputs; does not train, tune, or perform test inference. Run from repository root or notebook directory.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / "docs/phase-2/evaluation/test-metrics.json").exists():
    root = root.parent
assert (root / "docs/phase-2/evaluation/test-metrics.json").exists(), "Run from repo root or notebooks/"
metrics = json.loads((root / "docs/phase-2/evaluation/test-metrics.json").read_text())
protocol = json.loads((root / "docs/phase-2/evaluation/protocol.json").read_text())
print("Models:", list(protocol["models"]))
print("Top-level metric keys:", list(metrics))

## Frozen headline results

These figures were recorded by the evaluation team on the 377-image test set. The Phase 1 SVM here is the full-training refit, **not** the historical Phase 1 ensemble.

In [ ]:
labels = {"phase1_svm": "Phase 1 SVM (refit)", "a2_1_cnn": "A2.1 CNN", "h3_hybrid": "H3 hybrid"}
results = pd.DataFrame([
    {"Model": labels[key], "Macro F1": data["unconditional"]["macro_f1"],
     "Accuracy": data["unconditional"]["accuracy"],
     "ECE": data["unconditional"]["expected_calibration_error"]}
    for key, data in metrics["models"].items()
])
display(results.round(4))
ax = results.plot.bar(x="Model", y="Macro F1", legend=False, ylim=(0, 1), rot=0, title="Frozen test macro F1")
ax.set_ylabel("Macro F1")
plt.tight_layout()
plt.show()

## Pre-registered demo thresholds

Read directly from the frozen protocol; no re-optimization.

In [ ]:
thresholds = pd.DataFrame([
    {"Model key": key, "Temperature": item["temperature"], "Demo threshold": item["demo_threshold"], "Headline threshold": item["headline_threshold"]}
    for key, item in protocol["models"].items()
])
display(thresholds)

## McNemar interpretation and limitations

- H3 vs A2.1: exact p ≈ 0.000019, significant at Bonferroni α = 0.0167.
- H3 vs refit SVM: p ≈ 0.0351, **not** significant after correction.
- The 12-image Grad-CAM sample cannot establish background robustness.
- No external robustness set was collected.

In [ ]:
from IPython.display import display, Image
for filename in ["test-confusion-h3_hybrid.png", "test-reliability-h3_hybrid.png", "test-gradcam-a2_1_cnn.png"]:
    path = root / "docs/phase-2/evaluation" / filename
    if path.exists():
        print(filename); display(Image(filename=str(path), width=680))